## Etapa 6 — Construção e Validação da ABT PGFN (2024–2025)

Este notebook valida a ABT construída a partir dos dados da PGFN
(SIDA não previdenciário), com foco em Pessoa Jurídica (PJ), garantindo:

- integridade do identificador CNPJ;
- consistência dos domínios categóricos;
- coerência temporal e numérica;
- aderência às decisões de modelagem da ABT.

A validação é feita por amostragem de *row groups* do Parquet, evitando
leitura integral do dataset.


### Setup e caminhos (Code)

In [3]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

PROJECT_ROOT = Path("..")
ABT_PATH = PROJECT_ROOT / "data" / "processed" / "pgfn" / "pgfn_abt_2024_2025.parquet"

assert ABT_PATH.exists(), f"Arquivo não encontrado: {ABT_PATH}"



## Metadata do Parquet (Code)

In [4]:
pf = pq.ParquetFile(ABT_PATH)

print("Arquivo:", ABT_PATH.name)
print("Total de linhas:", pf.metadata.num_rows)
print("Row groups:", pf.num_row_groups)
print("\nSchema:")
print(pf.schema)


Arquivo: pgfn_abt_2024_2025.parquet
Total de linhas: 143422554
Row groups: 10959

Schema:
required group field_id=-1 schema {
  optional binary field_id=-1 cnpj (String);
  optional binary field_id=-1 cnpj_raiz (String);
  optional binary field_id=-1 tipo_devedor (String);
  optional binary field_id=-1 numero_inscricao (String);
  optional binary field_id=-1 situacao_inscricao (String);
  optional binary field_id=-1 tipo_situacao_inscricao (String);
  optional binary field_id=-1 receita_principal (String);
  optional int64 field_id=-1 data_inscricao (Timestamp(isAdjustedToUTC=false, timeUnit=milliseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int32 field_id=-1 indicador_ajuizado (Int(bitWidth=8, isSigned=true));
  optional double field_id=-1 valor_consolidado;
  optional binary field_id=-1 unidade_responsavel (String);
  optional int32 field_id=-1 ano;
  optional int32 field_id=-1 trimestre;
  optional binary field_id=-1 nome_devedor (String);
}



A ABT consolidada para 2024–2025 possui alta volumetria
(143.422.554 registros), organizada em blocos que permitem leitura
eficiente sem carregamento integral em memória.

### Dicionário resumido da ABT


| Variável | Tipo / Codificação | Descrição |
|--------|------------------|----------|
| cnpj | String (14 dígitos) | CNPJ completo da empresa devedora |
| cnpj_raiz | String (8 dígitos) | Raiz do CNPJ, correspondente aos 8 primeiros dígitos |
| tipo_devedor | String | Tipo do devedor na inscrição (ex.: principal) |
| numero_inscricao | String | Identificador da inscrição em dívida ativa |
| situacao_inscricao | String | Situação original da inscrição conforme base da PGFN |
| tipo_situacao_inscricao | String (categorias) | Situação normalizada da inscrição |
| receita_principal | String | Tipo de receita associada à inscrição |
| data_inscricao | Data | Data de inscrição do débito |
| indicador_ajuizado | Binária (0/1) | 1 = inscrição ajuizada; 0 = não ajuizada |
| valor_consolidado | Numérica contínua | Valor consolidado do débito |
| unidade_responsavel | String | Unidade da PGFN responsável |
| ano | Inteiro | Ano de referência do arquivo |
| trimestre | Inteiro (1–4) | Trimestre de referência do arquivo |
| nome_devedor | String | Razão social da empresa |


### Visualização inicial (head) (Code)

In [5]:
table_head = pf.read_row_group(0)
df_head = table_head.to_pandas()

df_head.head(10)


,cnpj,cnpj_raiz,tipo_devedor,numero_inscricao,situacao_inscricao,tipo_situacao_inscricao,receita_principal,data_inscricao,indicador_ajuizado,valor_consolidado,unidade_responsavel,ano,trimestre,nome_devedor
0,28196878000163,28196878,PRINCIPAL,2172000069480,ATIVA EM COBRANCA,irregular,Receita da dívida ativa - PIS,2020-05-11,0,5.823400e+04,AMAZONAS,2024,1,MORESCHI CLINICA ODONTOLOGICA LTDA
1,84461748000181,84461748,PRINCIPAL,2179900094488,ATIVA AJUIZADA,irregular,Receita da dívida ativa - PIS,1999-11-30,1,1.289772e+07,AMAZONAS,2024,1,ACBR COMPUTADORES DA AMAZONIA LTDA
2,84661206000152,84661206,PRINCIPAL,2170800004092,ATIVA AJUIZADA,irregular,Receita da dívida ativa - PIS,2008-04-14,1,5.550758e+06,AMAZONAS,2024,1,CAPA CONSTRUCOES R PAVIMENTACAO LTDA
3,28196878000163,28196878,PRINCIPAL,2162000543834,ATIVA EM COBRANCA,irregular,Receita da dívida ativa - CSLL,2020-05-11,0,2.824280e+05,AMAZONAS,2024,1,MORESCHI CLINICA ODONTOLOGICA LTDA
4,84461748000181,84461748,PRINCIPAL,2169900401936,ATIVA AJUIZADA,irregular,Receita da dívida ativa - CSLL,1999-11-30,1,3.924561e+08,AMAZONAS,2024,1,ACBR COMPUTADORES DA AMAZONIA LTDA
5,14104972000130,14104972,PRINCIPAL,7061401346234,ATIVA AJUIZADA,irregular,Receita da dívida ativa - CSLL,2014-03-07,1,2.232669e+06,AMAZONAS,2024,1,RJ FONTES CONSULTORIA EM ENGENHARIA LTDA
6,84461748000181,84461748,PRINCIPAL,2149700011889,ATIVA COM AJUIZAMENTO A SER PROSSEGUIDO,irregular,Receita da dívida ativa - Imposto de Importação,1997-09-16,1,5.591190e+07,AMAZONAS,2024,1,ACBR COMPUTADORES DA AMAZONIA LTDA
7,28196878000163,28196878,PRINCIPAL,2122000134390,ATIVA EM COBRANCA,irregular,Receita da dívida ativa - IRPJ,2020-05-11,0,4.707150e+05,AMAZONAS,2024,1,MORESCHI CLINICA ODONTOLOGICA LTDA
8,84461748000181,84461748,PRINCIPAL,2129900183735,ATIVA COM AJUIZAMENTO A SER PROSSEGUIDO,irregular,Receita da dívida ativa - IRPJ,1999-11-30,1,1.408339e+09,AMAZONAS,2024,1,ACBR COMPUTADORES DA AMAZONIA LTDA
9,14104972000130,14104972,PRINCIPAL,7021400512379,ATIVA AJUIZADA,irregular,Receita da dívida ativa - IRPJ,2014-03-07,1,3.919052e+06,AMAZONAS,2024,1,RJ FONTES CONSULTORIA EM ENGENHARIA LTDA


A visualização inicial confirma que a ABT está organizada no nível de
inscrição em dívida ativa, corretamente associada a empresas por CNPJ.
Observa-se a recorrência de múltiplas inscrições para o mesmo CNPJ,
o que valida a escolha do modelo relacional e antecipa a necessidade
de agregação por empresa nas próximas etapas.



### Amostragem controlada para validações (Code)

In [6]:
def is_digits_len(s: str, n: int) -> bool:
    return bool(re.fullmatch(rf"\d{{{n}}}", str(s)))

# seleciona alguns row groups distribuídos
rg_ids = np.linspace(0, pf.num_row_groups - 1, num=20, dtype=int)

cols_check = [
    "cnpj",
    "cnpj_raiz",
    "tipo_devedor",
    "situacao_inscricao",
    "tipo_situacao_inscricao",
    "indicador_ajuizado",
    "ano",
    "trimestre",
    "data_inscricao",
    "valor_consolidado"
]

tables = [pf.read_row_group(int(i), columns=cols_check) for i in rg_ids]
df = pa.concat_tables(tables).to_pandas()

df.shape


(261526, 10)

A amostra utilizada contém 261.526 registros e 10 variáveis, sendo
suficiente para validar a estrutura, os domínios e a integridade dos
principais campos da ABT, sem necessidade de leitura integral da base.


### Integridade do CNPJ (Code)

In [7]:
df["cnpj_ok"] = df["cnpj"].apply(lambda x: is_digits_len(x, 14))
df["raiz_ok"] = df["cnpj_raiz"].apply(lambda x: is_digits_len(x, 8))
df["raiz_bate"] = df.apply(
    lambda r: str(r["cnpj"])[:8] == str(r["cnpj_raiz"]),
    axis=1
)

summary_cnpj = {
    "linhas_amostra": len(df),
    "cnpj_ok_%": round(df["cnpj_ok"].mean() * 100, 3),
    "cnpj_raiz_ok_%": round(df["raiz_ok"].mean() * 100, 3),
    "raiz_bate_%": round(df["raiz_bate"].mean() * 100, 3),
    "nulos_cnpj": int(df["cnpj"].isna().sum()),
    "nulos_raiz": int(df["cnpj_raiz"].isna().sum())
}

summary_cnpj


{'linhas_amostra': 261526,
 'cnpj_ok_%': np.float64(100.0),
 'cnpj_raiz_ok_%': np.float64(100.0),
 'raiz_bate_%': np.float64(100.0),
 'nulos_cnpj': 0,
 'nulos_raiz': 0}

Na amostra avaliada, os campos de CNPJ e raiz do CNPJ apresentaram
100% de conformidade, sem registros nulos ou inconsistências, o que
confirma a integridade do identificador empresarial da ABT.


### Domínios e sanidade dos dados (Code)

In [8]:
display(df["tipo_situacao_inscricao"].value_counts(dropna=False))
display(df["indicador_ajuizado"].value_counts(dropna=False))
display(df[["ano", "trimestre"]].value_counts())

print("Data inscrição (min/max):",
      df["data_inscricao"].min(),
      df["data_inscricao"].max())

df["valor_consolidado"].describe(
    percentiles=[0.5, 0.9, 0.99]
)


tipo_situacao_inscricao
irregular            230667
beneficio_fiscal      30347
garantia                292
suspenso_judicial       220
Name: count, dtype: int64

indicador_ajuizado
0    195318
1     66208
Name: count, dtype: int64

ano   trimestre
2024  3            48406
2025  1            45559
      2            41940
      3            38302
2024  1            37554
      4            25282
      2            24483
Name: count, dtype: int64

Data inscrição (min/max): 1991-04-04 00:00:00 2025-09-29 00:00:00


count    2.615260e+05
mean     1.148966e+07
std      2.152189e+08
min      1.874000e+03
50%      3.839365e+05
90%      6.610133e+06
99%      1.527886e+08
max      3.990464e+10
Name: valor_consolidado, dtype: float64

A distribuição das situações confirma o predomínio de inscrições em
situação irregular, com presença consistente de casos ajuizados e não
ajuizados. As variáveis temporais estão coerentes com o recorte
2024–2025, e os valores monetários apresentam distribuição assimétrica
esperada, com poucos casos de valores muito elevados.


### Conclusão

A ABT PGFN 2024–2025 mostrou-se consistente na amostra avaliada.

- O identificador **CNPJ** apresenta formato válido e coerência com a
  respectiva **raiz do CNPJ**.
- Os domínios das variáveis categóricas e temporais estão alinhados às
  decisões de modelagem definidas na Etapa 6.
- Os valores monetários apresentam distribuição esperada, sem indícios
  de quebras estruturais na amostra.

A ABT está, portanto, apta para integração com a base externa de CNPJ,
permitindo a construção da base consolidada por empresa.
